> **Local runner:** Creates audit figures and evidence records from the saved aggregate CSVs. It does not fit models or generate patient assignments.

# Build figures and evidence records

The script reads the available primary and reconstruction summary tables. Missing patient-level evidence is not inferred from the plots.

In [ ]:
from pathlib import Path

def repository_root():
    for folder in (Path.cwd(), *Path.cwd().parents):
        if (folder / "scripts" / "validate_experiment_evidence.py").is_file():
            return folder
    raise FileNotFoundError("Open Jupyter from inside the repository directory")

ROOT = repository_root()
import sys
SCRIPT = ROOT / "scripts" / "build_audit_report.py"
PRIMARY = ROOT / "evidence" / "primary_candidates_MANUSCRIPT_TABLE_II_ONLY.csv"
RECON = ROOT / "evidence" / "reconstruction"
OUTPUT = ROOT / "outputs" / "audit_report"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Script:", SCRIPT)
print("Summary sources:", PRIMARY, RECON)

## Generate the figures and audit records

Runs the included reporting script with your notebook's Python environment. Figures are saved as 300-dpi PNG and SVG in `output`. Re-running overwrites the reporting outputs but **does not alter** the clustering notebooks or source input CSVs.

In [ ]:
import subprocess
cmd = [
    sys.executable, str(SCRIPT),
    "--primary", str(PRIMARY),
    "--restart", str(RECON / "restart_budget_comparison.csv"),
    "--cross-weight", str(RECON / "cross_weight_partition_agreement_ninit20_REFERENCE.csv"),
    "--multistart", str(RECON / "weight_transition_summary_ninit20.csv"),
    "--distance", str(RECON / "distance_and_negative_control.csv"),
    "--out", str(OUTPUT),
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError(f"Audit reporting failed ({result.returncode})")

## Display the four generated figures inside the notebook

Images are shown inline using the saved PNGs. The Python report script uses a noninteractive graphics backend, so Jupyter will **not** display plots automatically when the script finishes; the following cell is how to view them.

In [ ]:
from IPython.display import display, Image, Markdown

figures = [
    ("Primary five-candidate reliability audit", "Fig_Audit_PrimaryCandidates.png"),
    ("Restart stability: 1 vs 20 internal restarts", "Fig_Reconstruction_RestartStability.png"),
    ("Reconstruction cross-weight ARI", "Fig_Reconstruction_CrossWeightARI.png"),
    ("Reconstruction distance shares (not primary 06B)", "Fig_Reconstruction_DistanceShares.png"),
]
for title, filename in figures:
    path = OUTPUT / filename
    print(f"\n{title}\n{path}")
    if path.is_file():
        display(Image(filename=str(path), width=950))
    else:
        print("Not found. Check the execution log above.")

## Inspect machine-readable evidence and export paths

The records combine the **five manuscript-reported primary candidates** with **one separate reconstruction record**. They are not six independent models and do not establish clinical validity.

In [ ]:
import pandas as pd

audit = pd.read_csv(OUTPUT / "claim_aware_evidence_records.csv")
display(audit[["candidate_id", "evidence_stream", "review_flags", "review_disposition"]])
print("\nFiles created:")
for file in sorted(OUTPUT.iterdir()):
    print(" •", file.name)
print("\nFigure captions:")
print((OUTPUT / "figure_captions.txt").read_text(encoding="utf-8"))

## Optional: patient transition matrix for weights 0.600 vs 1.000

**Do not run this until you have real patient-level labels** from the reconstruction. Export two CSV files for exactly the same patient cohort, each with columns `patient_nbr,label`. Assignments must correspond to the selected minimum-inertia fitted models. Place them in `inputs`, then set `MAKE_TRANSITION=True`. This does not generate synthetic labels.

In [ ]:
MAKE_TRANSITION = False  # Change to True only after exporting BOTH real label CSVs.
COHORT = "development"  # Set to "heldout" for held-out files, with matching filenames.

if MAKE_TRANSITION:
    a = (ROOT / "outputs" / "labels") / f"R2_robust_w060_{COHORT}_labels.csv"
    b = (ROOT / "outputs" / "labels") / f"R2_robust_w100_{COHORT}_labels.csv"
    assert a.is_file() and b.is_file(), f"Missing label exports: {a} / {b}"
    cmd_with_labels = cmd + ["--labels-a", str(a), "--labels-b", str(b), "--label-cohort", COHORT]
    subprocess.run(cmd_with_labels, check=True)
    display(Image(filename=str(OUTPUT / f"Fig_PatientTransitions_{COHORT}.png"), width=800))
else:
    print("Patient transition matrix skipped. No patient-level labels were inferred.")